# 01 — Environment Setup

**Objective**: verify the Python environment, package versions, `.env` loading, config files, and sample data are all in place before any connector work (Phases 3+) begins.

**Dependencies**: `requirements.txt` installed in the active environment.

**Configuration**: reads `.env` if present (optional at this stage — no live credentials are required for Phase 1).

In [1]:
import sys
import importlib

REQUIRED = ["pydantic", "yaml", "pandas", "dotenv"]
OPTIONAL = ["langchain", "langgraph", "mem0", "streamlit"]  # land in later phases

print(f"Python: {sys.version}\n")
for mod in REQUIRED:
    try:
        m = importlib.import_module(mod)
        print(f"  OK       {mod:12s} {getattr(m, '__version__', '(no __version__)')}")
    except ImportError as e:
        print(f"  MISSING  {mod:12s} -> {e}")

for mod in OPTIONAL:
    try:
        m = importlib.import_module(mod)
        print(f"  OK       {mod:12s} {getattr(m, '__version__', '(no __version__)')} (not required until a later phase)")
    except ImportError:
        print(f"  not installed yet: {mod:12s} (expected — installed when its phase lands)")

Python: 3.14.7 (main, Aug 18 2026, 17:53:12) [Clang 14.0.3 (clang-1403.0.22.14.1)]

  OK       pydantic     2.13.5
  OK       yaml         6.0.3


  OK       pandas       3.0.5
  OK       dotenv       (no __version__)
  not installed yet: langchain    (expected — installed when its phase lands)
  OK       langgraph    (no __version__) (not required until a later phase)


  OK       mem0         2.0.19 (not required until a later phase)


  OK       streamlit    1.63.0 (not required until a later phase)


In [2]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")  # fine if missing — nothing in Phase 1 requires real creds

import os
for key in ["JIRA_URL", "JIRA_EMAIL", "JIRA_API_TOKEN", "OPENAI_API_KEY", "MEM0_API_KEY"]:
    present = bool(os.getenv(key))
    print(f"  {'set' if present else 'not set':7s}  {key}")

  not set  JIRA_URL
  not set  JIRA_EMAIL
  not set  JIRA_API_TOKEN
  not set  OPENAI_API_KEY
  not set  MEM0_API_KEY


## Config files present

In [3]:
import yaml

config_dir = PROJECT_ROOT / "config"
for name in ["project_mapping.yaml", "status_mapping.yaml", "risk_rules.yaml"]:
    path = config_dir / name
    assert path.exists(), f"missing {path}"
    with open(path) as f:
        data = yaml.safe_load(f)
    print(f"  OK  {name:24s} top-level keys: {list(data.keys())}")

  OK  project_mapping.yaml     top-level keys: ['organization_id', 'portfolio_id', 'projects']
  OK  status_mapping.yaml      top-level keys: ['status_map', 'blocker_status_map', 'resolved_statuses', 'default_unmapped_status']
  OK  risk_rules.yaml          top-level keys: ['delivery_risk', 'financial_risk', 'combined_risk_matrix', 'reason_codes', 'freshness_thresholds_hours', 'trend_thresholds']


## Sample data present and readable

In [4]:
import pandas as pd

jira_df = pd.read_csv(PROJECT_ROOT / "data/sample/jira_mock_data.csv")
fin_df = pd.read_csv(PROJECT_ROOT / "data/sample/financial_mock_data.csv")

print("jira_mock_data.csv:", jira_df.shape, "projects:", sorted(jira_df["project_key"].unique()))
print("financial_mock_data.csv:", fin_df.shape, "projects:", sorted(fin_df["project_id"].unique()))

jira_mock_data.csv: (270, 27) projects: ['LYNX', 'NOVA', 'ORCA', 'PHX', 'QSR', 'TITAN']
financial_mock_data.csv: (43, 17) projects: [np.int64(10001), np.int64(10002), np.int64(10003), np.int64(10004), np.int64(10005), np.int64(10007)]


## Pydantic models importable and constructible

In [5]:
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import Project, FinancialRecord, RAGStatus
from datetime import date

fr = FinancialRecord(
    project_id="10001",
    reporting_period="2026-03",
    approved_budget=182342.49,
    actual_spend=203307.39,
    committed_spend=213093.97,
    forecast_spend=206329.15,
).with_calculated_fields()

print(fr.model_dump())

{'project_id': '10001', 'reporting_period': '2026-03', 'approved_budget': 182342.49, 'actual_spend': 203307.39, 'committed_spend': 213093.97, 'forecast_spend': 206329.15, 'currency': 'USD', 'remaining_budget': -234058.87000000002, 'budget_consumption_pct': 111.49753960253588, 'forecast_variance': -23986.660000000003, 'source_reported_remaining_budget': None, 'retrieved_timestamp': datetime.datetime(2026, 9, 3, 3, 22, 27, 748142, tzinfo=datetime.timezone.utc)}


## Validation checks

- [ ] Required packages import cleanly
- [ ] `.env` loads without error (values may be unset at this phase)
- [ ] All three config YAML files parse
- [ ] Both sample CSVs load and contain the expected 6 projects (5 mapped + 1 deliberately unmapped, `QSR`)
- [ ] `FinancialRecord.with_calculated_fields()` runs the Section 5 formulas without raising

## Error handling

Every check above prints its own pass/fail line rather than crashing the
notebook on the first missing dependency, so a partial environment is still
actionable feedback rather than a stack trace.

## Testing

This notebook *is* the smoke test for Phase 1 — see `tests/README.md` for
where unit tests land starting Phase 6.

## Next step

`02_jira_connection.ipynb` (Phase 3) — implement `src/connectors/jira_client.py`
against a real or sandbox Jira Cloud instance.